In [2]:
import langchain
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent
import os
from pydantic import BaseModel, Field, field_validator
import json
from pathlib import Path
from typing import List
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import numpy as np

In [4]:
# API key
load_dotenv()
os.environ["GROQ_API_KEY"] = "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"
print(os.getenv("GROQ_API_KEY"))

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


# 1. Get data query data

In [5]:
# Get data from the query-p1-groupA folder
CURRENT_PATH = os.getcwd()
print(CURRENT_PATH)
ROOT_PATH = os.path.abspath(os.path.join(CURRENT_PATH, "../../"))
QUERY_PATH = os.path.join(ROOT_PATH, "scripts", "query-p1-groupA")
print(QUERY_PATH)

d:\University\Projects\Individual projects\Multimodal-Retrieval\notebooks\agent
d:\University\Projects\Individual projects\Multimodal-Retrieval\scripts\query-p1-groupA


Đọc data từ file query

In [6]:
def read_query_file(file_path: str | Path) -> str:
    """Read one UTF-8 query from a text file."""
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {path}")

    if not path.is_file():
        raise ValueError(f"Đường dẫn không phải file: {path}")

    query = path.read_text(encoding="utf-8-sig").strip()

    if not query:
        raise ValueError(f"File query rỗng: {path}")

    return query


def _deduplicate_queries(queries: List[str]) -> List[str]:
    result: List[str] = []
    seen: set[str] = set()

    for item in queries:
        query = " ".join(item.strip().split())
        key = query.casefold()

        if query and key not in seen:
            result.append(query)
            seen.add(key)

    return result

# 2. Structured output

In [7]:
class QueryExpansionOutput(BaseModel):
    """Structured output returned by the query-expansion model."""

    queries: List[str] = Field(
        description=(
            "A list of unique, self-contained English search queries "
            "that preserve the facts in the original query."
        )
    )

    @field_validator("queries")
    @classmethod
    def clean_queries(cls, values: List[str]) -> List[str]:
        cleaned: List[str] = []
        seen: set[str] = set()

        for value in values:
            query = " ".join(value.strip().split())
            key = query.casefold()

            if query and key not in seen:
                cleaned.append(query)
                seen.add(key)

        if not cleaned:
            raise ValueError("The model returned no usable query.")

        return cleaned


# 3. Model

Note: Using few shot prompt instead of fine-tuned

Chạy test model gpt-oss-120b tối ưu system prompt, hyperparameter

In [8]:
MODEL_NAME = "openai/gpt-oss-120b"

model = ChatGroq(
    model=MODEL_NAME,
    temperature=0.4,
    reasoning_effort="medium",
    max_tokens=2048,
    timeout=60,
    max_retries=3,
)

structured_llm = model.with_structured_output(
    QueryExpansionOutput,
    method="json_schema",
    strict=True,
)

SYSTEM_PROMPT = """
You are a query-expansion planner for a large-scale multimedia retrieval system.

The input query can be written in Vietnamese. Generate search-ready queries in English.

Requirements:
- Produce exactly {k} UNIQUE expanded queries.
- Preserve every reliable fact from the original query.
- Do not invent names, dates, organizations, spacecraft, missions, people, or locations
  that are not explicitly present in the original query.
- Make every query self-contained so it can be searched independently.
- Diversify the retrieval intent across useful multimedia evidence, such as:
  1. visual appearance and objects,
  2. spoken narration, subtitles, or ASR transcript,
  3. event or activity description,
  4. entities and relations,
  5. concise lexical or keyword matching.
- Use natural English, not a literal word-for-word translation.
- Do not answer the query.
- Return only data matching the required structured schema.
""".strip()

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        (
            "human",
            """Original query:
{query}

Number of expanded queries required: {k}
""",
        ),
    ]
)

expansion_chain = prompt | structured_llm

In [9]:
def expand_query(
    query: str,
    k: int = 5,
    max_attempts: int = 3,
) -> List[str]:
    """Generate exactly k unique English search queries."""
    if not query.strip():
        raise ValueError("Query không được để trống.")

    if k < 1:
        raise ValueError("k phải lớn hơn hoặc bằng 1.")

    if k > 20:
        raise ValueError("Notebook giới hạn k <= 20 để tránh output quá dài.")

    collected: List[str] = []
    current_instruction = query.strip()

    for _ in range(max_attempts):
        result = expansion_chain.invoke(
            {
                "query": current_instruction,
                "k": k,
            }
        )
        # Xóa những query trùng lặp và thêm vào danh sách collected -> lọc lần 1
        collected = _deduplicate_queries(collected + result.queries)

        # LLM sinh quá nhiều query -> lọc lần 2
        if len(collected) >= k:
            return collected[:k]

        missing = k - len(collected)
        current_instruction = (
            f"{query.strip()}\n\n"
            f"The following expanded queries already exist and MUST NOT be repeated:\n"
            + "\n".join(f"- {item}" for item in collected)
            + f"\nGenerate {missing} additional distinct search queries."
        )

    raise RuntimeError(
        f"Model chỉ tạo được {len(collected)}/{k} query duy nhất "
        f"sau {max_attempts} lần gọi."
    )

# Format lại output
def expand_query_from_file(
    file_path: str | Path,
    k: int = 5,
) -> dict:
    """Read a query file and return the original plus k expansions."""
    original_query = read_query_file(file_path)
    expanded_queries = expand_query(original_query, k=k)

    return {
        "source_file": str(Path(file_path)),
        "model": MODEL_NAME,
        "k": k,
        "original_query": original_query,
        "expanded_queries": expanded_queries,
    }


# 4. Run demo ^^

* Chạy unit test 
* Đo similarity bằng model embedding mạnh hơn - nghiên cứu thử xem top model hiện nay + free

In [10]:
QUERY_FILE = os.path.join(QUERY_PATH, "query-p1-1-kis.txt")
query = read_query_file(QUERY_FILE)
query

'Đây là phần giới thiệu việc phóng tàu vũ trụ tư nhân. Đoạn clip bắt đầu với hình ảnh 4 phi hành gia mặc áo đen. Một trong những nhiệm vụ dự kiến của tàu vũ trụ là nghiên cứu ánh sáng cực quang ở vùng cực'

## 4.1. Đo similarity unit test

Sinh thử k = 5 bản expanded query và so similarity demo

### Unit Test 1

In [11]:
K = 5
result = expand_query_from_file(
    file_path=QUERY_FILE,
    k=K,
)
for index, query in enumerate(result["expanded_queries"], start=1):
    print(f"{index}. {query}")

1. video introduction of a private spaceflight launch showing four astronauts wearing black suits
2. audio narration describing a private spacecraft mission that will study polar aurora lighting
3. clip that begins with four black‑suited astronauts as part of a private launch and mentions research of auroral light in the polar region
4. information about a private space mission planned to investigate aurora phenomena at the Earth's poles
5. search keywords: private space launch, four astronauts, black suits, aurora research, polar region


In [12]:
result

{'source_file': 'd:\\University\\Projects\\Individual projects\\Multimodal-Retrieval\\scripts\\query-p1-groupA\\query-p1-1-kis.txt',
 'model': 'openai/gpt-oss-120b',
 'k': 5,
 'original_query': 'Đây là phần giới thiệu việc phóng tàu vũ trụ tư nhân. Đoạn clip bắt đầu với hình ảnh 4 phi hành gia mặc áo đen. Một trong những nhiệm vụ dự kiến của tàu vũ trụ là nghiên cứu ánh sáng cực quang ở vùng cực',
 'expanded_queries': ['video introduction of a private spaceflight launch showing four astronauts wearing black suits',
  'audio narration describing a private spacecraft mission that will study polar aurora lighting',
  'clip that begins with four black‑suited astronauts as part of a private launch and mentions research of auroral light in the polar region',
  "information about a private space mission planned to investigate aurora phenomena at the Earth's poles",
  'search keywords: private space launch, four astronauts, black suits, aurora research, polar region']}

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-m3")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 21465.89it/s]


In [18]:
def cosine_similarity(vec1, vec2):
    """Compute cosine similarity between two vectors."""
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def compute_similarity(original_queries: list[str], expanded_queries: list[str]) -> np.ndarray:
    """Compute similarity between original queries and expanded queries."""
    original_embeddings = embedder.encode(original_queries)
    expanded_embeddings = embedder.encode(expanded_queries)

    similarity_scores = []
    for orig_emb in original_embeddings:
        row = []
        for exp_emb in expanded_embeddings:
            row.append(cosine_similarity(orig_emb, exp_emb))
        similarity_scores.append(row)

    return np.array(similarity_scores)

In [19]:
# Đo similarity giữa các original query và expanded query
original_queries = [result["original_query"]]
expanded_queries = result["expanded_queries"]

original_embeddings = embedder.encode(original_queries)
expanded_embeddings = embedder.encode(expanded_queries)

similarity_scores = compute_similarity(original_queries, expanded_queries)

In [20]:
similarity_scores

array([[0.7052872 , 0.61096203, 0.72468233, 0.53744143, 0.5674646 ]],
      dtype=float32)

### Unit Test 2

In [15]:
K = 5
result2 = expand_query_from_file(
    file_path=QUERY_FILE,
    k=K,
)
for index, query in enumerate(result2["expanded_queries"], start=1):
    print(f"{index}. {query}")

1. private spaceflight introduction video showing four astronauts in black suits
2. clip beginning with four black‑suited astronauts visual scene
3. narration about a private spacecraft mission to study polar aurora
4. private spacecraft planned research on aurora borealis in polar region
5. keywords: private space launch, four astronauts, black suits, aurora research


### Unit Test 3

In [38]:
K = 5
result3 = expand_query_from_file(
    file_path=QUERY_FILE,
    k=K,
)
for index, query in enumerate(result3["expanded_queries"], start=1):
    print(f"{index}. {query}")

1. private spaceflight launch introduction video showing four astronauts wearing black suits
2. clip opening with four black‑suited astronauts and mentioning a mission to study polar aurora lights
3. visual footage of a private spacecraft launch featuring four astronauts in black suits at the start
4. audio narration describing a private spacecraft’s planned research of aurora illumination in the polar region
5. keyword search for private space launch, four astronauts, black suits, aurora research, polar area


# 5. Experiment

- Experimental objectives:
    1. Tối ưu system prompt, structured output, hyperparameters
    2. Tối ưu pipeline agent trong quá trình expand

- Experimental setup:
    - Models
    - Data
    - Metric: logic + system metric
    - System prompt
    - Hyperparameters:
        - temperatures 
        - reasoning_efforts
        - k_values
        - max_token_values
    - Workflow

- Result (Ghi thành cái bảng cho dễ so sánh)

## 5.1. Research question

Q1: LLM có tuân thủ yêu cầu không?

Q2: Query expansion có giữ đúng ý nghĩa không?

Q3: Query expansion có đủ đa dạng không? (khi có pipeline hoàn chỉnh thì test cái này)

Q4: Cùng một cấu hình có duy trì chất lượng ổn định qua nhiều lần chạy không?

Q5: Có đảm bảo chất lượng về latency, token usage?